In [ ]:
from  time_slice import time_slice
from read_edf import read_edf
from findpeaks import findpeaks
from scipy import signal
import matplotlib.pyplot as plt
import biosppy
import numpy as np
from signalavergedecg import signalavergedecg
from compute_late_potentials_from_avg import compute_late_potentials_from_avg
from waveletscaleaogram import waveletscaleaogram
from compare_signals_mannwhitney import compare_signals_mannwhitney
import os
from signaladd import signaladd
from iqr_winsorize import iqr_winsorize
import pprint
from tqdm import tqdm 
from scipy import stats
from calculate_band_power_ratio import calculate_band_power_ratio
import pandas as pd
import pprint
def main():
    save_dir_pic = "result/picture"
    os.makedirs(save_dir_pic, exist_ok=True)
    save_dir_txt = "result/txt"
    os.makedirs(save_dir_txt, exist_ok=True)
    save_dir_xlsx = "result/xlsx"
    os.makedirs(save_dir_xlsx, exist_ok=True)
    chanel_d,time,fs1,anot=read_edf("12_2_Not_filtered.edf","D:/ECG_IAI_RAS/RAT_NEW/12/2_rat/",[0, 1, 2, 3, 4, 5])
    sig1=chanel_d.get('II_LF           ')
    lf_sig,hf_sig,sig1,fs1,anot=signaladd("12_2_Not_filtered.edf","D:/ECG_IAI_RAS/RAT_NEW/12/2_rat/")
    sig1=iqr_winsorize(sig1,5)
   
    print('Длительность,с')
    print(len(sig1)/fs1)
    print(anot)
    
    if not anot:
        stabilization_time=0
        ishemia_time=1688
        reperfusion_time=3600
    else:
        stabilization_time=0
        ishemia_time=next((k for k, v in anot.items() if v == 'Ishemia'), 0)
        reperfusion_time=next((k for k, v in anot.items() if v == 'Reperfusion'), 0)

    ynorm=time_slice(
        sig1 - np.mean(sig1),
        stabilization_time+5,
        stabilization_time+34,
        fs1
    )

    ypat=time_slice(
        sig1 - np.mean(sig1),
        ishemia_time,
        ishemia_time+34,
        fs1
    )
    
    plt.figure(figsize=(15,7))
    plt.plot(np.linspace(0,len(ynorm)/fs1,len(ynorm)),ynorm)
    plt.grid(True)
    plt.savefig(os.path.join(save_dir_pic, "ecg_signal_norm.jpeg"), dpi=300)

    plt.figure(figsize=(15,7))
    plt.plot(np.linspace(0,len(ypat)/fs1,len(ypat)),ypat)
    plt.grid(True)
    plt.savefig(os.path.join(save_dir_pic, "ecg_signal_pat.jpeg"), dpi=300)
    rpeaks=findpeaks(ynorm,fs1)
    saecgnorm=signalavergedecg(ynorm,fs1,rpeaks)
    
    plt.figure(figsize=(15,7))
    plt.plot(np.linspace(0,len(saecgnorm)/fs1,len(saecgnorm)),saecgnorm)
    plt.grid(True)
    plt.savefig(os.path.join(save_dir_pic, "ecg_signal_averged_norm.jpeg"), dpi=300)
    parnorm=compute_late_potentials_from_avg(saecgnorm,fs1)
   

    Spnorm,frequenciesnorm,powernorm=waveletscaleaogram(saecgnorm,fs1)
    rationorm=calculate_band_power_ratio(powernorm,300,2000)
    print('Доля суммарной мощности в диапазоне 300-2000 Гц при норме ={rationorm}')
   
    plt.figure(figsize=(15,7))
    plt.subplot(121)
    time_axis = np.arange(powernorm.shape[1]) / fs1  
    plt.imshow(powernorm, 
           extent=[time_axis[0], time_axis[-1], frequenciesnorm[-1], frequenciesnorm[0]], 
           cmap='viridis', 
           aspect='auto',
           vmax=abs(powernorm).max(), 
           vmin=-abs(powernorm).max())  
    plt.colorbar(label='Мощность')
    plt.xlabel('Время (с)')
    plt.ylabel('Частота (Гц)')
    plt.title('Вейвлет-спектрограмма')
    plt.subplot(122)
    plt.semilogy(frequenciesnorm, Spnorm)
    plt.xlabel('Частота (Гц)')
    plt.ylabel('Суммарная мощность')
    plt.title('Интегральная мощность по частотам')
    plt.grid(True)
    
    plt.savefig(os.path.join(save_dir_pic, "ecg_signal_averged_norm_wavelet.jpeg"), dpi=300)
        
  
    rpeaks=findpeaks(ypat,fs1)
    saecgpat=signalavergedecg(ypat,fs1,rpeaks)
   
    plt.figure(figsize=(15,7))
    plt.plot(np.linspace(0,len(saecgpat)/fs1,len(saecgpat)),saecgpat)
    plt.grid(True)
    plt.savefig(os.path.join(save_dir_pic, "ecg_signal_averged_pat.jpeg"), dpi=300)

    Sppat,frequenciespat,powerpat=waveletscaleaogram(saecgpat,fs1)
    ratiopat=calculate_band_power_ratio(powerpat,300,2000)
    print('Доля суммарной мощности в диапазоне 300-2000 Гц при патологии ={ratiopat}')
   
    plt.figure(figsize=(15,7))
    plt.subplot(121)
    time_axis = np.arange(powerpat.shape[1]) / fs1  
    plt.imshow(powernorm, 
           extent=[time_axis[0], time_axis[-1], frequenciespat[-1], frequenciespat[0]], 
           cmap='viridis', 
           aspect='auto',
           vmax=abs(powerpat).max(), 
           vmin=-abs(powerpat).max())  
    plt.colorbar(label='Мощность')
    plt.xlabel('Время (с)')
    plt.ylabel('Частота (Гц)')
    plt.title('Вейвлет-спектрограмма')
    plt.subplot(122)
    plt.semilogy(frequenciespat, Sppat)
    plt.xlabel('Частота (Гц)')
    plt.ylabel('Суммарная мощность')
    plt.title('Интегральная мощность по частотам')
    plt.grid(True)
    
    plt.savefig(os.path.join(save_dir_pic, "ecg_signal_averged_pat_wavelet.jpeg"), dpi=300)

    parpat=compute_late_potentials_from_avg(saecgpat,fs1)
    print(parpat)

    result=compare_signals_mannwhitney(saecgnorm,saecgpat)
    print(result)
    # Инициализируем словарь с результатами для текущего файла
    file_result = {
    'rat_number': None,  # Явно сохраняем номер крысы
    # Параметры нормального сигнала
    'norm_fQRS_ms': None,
    'norm_RMS40_mkV': None,
    'norm_LAS_ms': None,
    # Параметры патологического сигнала
    'pat_fQRS_ms': None,
    'pat_RMS40_mkV': None,
    'pat_LAS_ms': None
    }
    base_path = "D:/ECG_IAI_RAS/RAT_NEW/"
    file_numbers = range(10, 19)
    results_data = []

    # Создаем прогресс-бар с дополнительной информацией
    pbar = tqdm(
    file_numbers, 
    desc="Обработка файлов", 
    unit="файл",
    ncols=100,  # ширина прогресс-бара
    colour='green',  # цвет (работает в некоторых терминалах)
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )
    for i in pbar:
        # Обновляем описание с текущим файлом
        pbar.set_description(f"Обработка файла {i}")
        file_result['rat_number'] =i
        # Здесь ваш код обработки
        print(f"\n{'='*50}")
        print(f"Обработка файла {i}")
        print('='*50)
    
        # Формируем пути к файлам
        file_name = f"{i}_2_Not_filtered.edf"
        file_path = f"{base_path}{i}/2_rat/"
        chanel_d, time, fs1, anot = read_edf(
            file_name, 
            file_path, 
            [0, 1, 2, 3, 4, 5]
        )
        
        pbar.set_postfix(
            file=file_name,
            status="Обработка сигнала..."
        )
        # Дополнительная обработка сигнала
        lf_sig, hf_sig, sig1, fs1, anot = signaladd(
            file_name, 
            file_path
        )
        
        # Применяем винзоризацию
        sig1 = iqr_winsorize(sig1, 5)
        
        
        # Определяем временные метки
        if not anot:
            stabilization_time = 0
            ishemia_time = 1688
            reperfusion_time = 3600
        else:
            stabilization_time = 0
            ishemia_time = next((k for k, v in anot.items() if v == 'Ishemia'), 0)
            reperfusion_time = next((k for k, v in anot.items() if v == 'Reperfusion'), 0)
        
        # Вырезаем участки сигнала
        ynorm = time_slice(
            sig1 - np.mean(sig1),
            stabilization_time + 5,
            stabilization_time + 34,
            fs1
        )
        
        ypat = time_slice(
            sig1 - np.mean(sig1),
            ishemia_time,
            ishemia_time + 34,
            fs1
        )
        
        pbar.set_postfix(
            file=file_name,
            status="Графики сигналов..."
        )
        # Сохраняем графики исходных сигналов
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(ynorm)/fs1, len(ynorm)), ynorm)
        plt.grid(True)
        plt.title(f'Нормальный сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_norm_2_{i}.jpeg"), dpi=300)
        plt.close()
        
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(ypat)/fs1, len(ypat)), ypat)
        plt.grid(True)
        plt.title(f'Патологический сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_pat_2_{i}.jpeg"), dpi=300)
        plt.close()
        
        pbar.set_postfix(
            file=file_name,
            status="Усреднение сигналов..."
        )
        
        # Обработка нормального сигнала
        rpeaks = findpeaks(ynorm, fs1)
        saecgnorm = signalavergedecg(ynorm, fs1, rpeaks)
        
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(saecgnorm)/fs1, len(saecgnorm)), saecgnorm)
        plt.grid(True)
        plt.title(f'Усредненный нормальный сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_norm_2_{i}.jpeg"), dpi=300)
        plt.close()
        
        parnorm = compute_late_potentials_from_avg(saecgnorm, fs1)
        # Сохраняем результаты
        with open(os.path.join(save_dir_txt, f"results_2_{i}_norm.txt"), 'w') as f:
            f.write(f"Результаты обработки файла {i}\n")
            f.write("="*50 + "\n")
          
            f.write(f"Нормальный сигнал:\n{parnorm}\n\n")
            f.close()
        # Сохраняем параметры в словарь
        if isinstance(parnorm, dict):
            file_result['norm_fQRS_ms'] = parnorm.get('fQRS_ms', None)
            file_result['norm_RMS40_mkV'] = parnorm.get('RMS40_mkV', None)
            file_result['norm_LAS_ms'] = parnorm.get('LAS_ms', None)
        elif isinstance(parnorm, (list, tuple)) and len(parnorm) >= 3:
            file_result['norm_fQRS_ms'] = parnorm[0]
            file_result['norm_RMS40_mkV'] = parnorm[1]
            file_result['norm_LAS_ms'] = parnorm[2]
        
        pbar.set_postfix(
            file=file_name,
            status="Вейвлет-анализ (норма)..."
        )
        
        # Вейвлет-анализ нормального сигнала
        Spnorm, frequenciesnorm, powernorm = waveletscaleaogram(saecgnorm, fs1)
        rationorm=calculate_band_power_ratio(powernorm,300,2000)
        with open(os.path.join(save_dir_txt, f"results_ratio_cwt_power_2_{i}_norm.txt"), 'w') as f:
            f.write(f"Результаты обработки файла {i}\n")
            f.write("="*50 + "\n")
          
            f.write(f"Нормальный сигнал:\n{rationorm}\n\n")
            f.close()
        
        plt.figure(figsize=(15,7))
        plt.subplot(121)
        time_axis = np.arange(powernorm.shape[1]) / fs1  
        plt.imshow(powernorm, 
               extent=[time_axis[0], time_axis[-1], frequenciesnorm[-1], frequenciesnorm[0]], 
               cmap='viridis', 
               aspect='auto',
               vmax=abs(powernorm).max(), 
               vmin=-abs(powernorm).max())  
        plt.colorbar(label='Мощность')
        plt.xlabel('Время (с)')
        plt.ylabel('Частота (Гц)')
        plt.title(f'Вейвлет-спектрограмма (норма) - Файл {i}')
        
        plt.subplot(122)
        plt.semilogy(frequenciesnorm, Spnorm)
        plt.xlabel('Частота (Гц)')
        plt.ylabel('Суммарная мощность')
        plt.title('Интегральная мощность по частотам')
        plt.grid(True)
        
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_norm_wavelet_2_{i}.jpeg"), dpi=300)
        plt.close()
        
        pbar.set_postfix(
            file=file_name,
            status="Обработка патологии..."
        )
        
        # Обработка патологического сигнала
        rpeaks = findpeaks(ypat, fs1)
        saecgpat = signalavergedecg(ypat, fs1, rpeaks)
        
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(saecgpat)/fs1, len(saecgpat)), saecgpat)
        plt.grid(True)
        plt.title(f'Усредненный патологический сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_pat_2_{i}.jpeg"), dpi=300)
        plt.close()
        
        pbar.set_postfix(
            file=file_name,
            status="Вейвлет-анализ (патология)..."
        )
        
        # Вейвлет-анализ патологического сигнала
        Sppat, frequenciespat, powerpat = waveletscaleaogram(saecgpat, fs1)
        ratiopat=calculate_band_power_ratio(powerpat,300,2000)
        with open(os.path.join(save_dir_txt, f"results_ratio_cwt_power_2_{i}_pat.txt"), 'w') as f:
            f.write(f"Результаты обработки файла {i}\n")
            f.write("="*50 + "\n")
          
            f.write(f"Нормальный сигнал:\n{ratiopat}\n\n")
            f.close()
        
        plt.figure(figsize=(15,7))
        plt.subplot(121)
        time_axis = np.arange(powerpat.shape[1]) / fs1  
        plt.imshow(powerpat, 
               extent=[time_axis[0], time_axis[-1], frequenciespat[-1], frequenciespat[0]], 
               cmap='viridis', 
               aspect='auto',
               vmax=abs(powerpat).max(), 
               vmin=-abs(powerpat).max())  
        plt.colorbar(label='Мощность')
        plt.xlabel('Время (с)')
        plt.ylabel('Частота (Гц)')
        plt.title(f'Вейвлет-спектрограмма (патология) - Файл {i}')
        
        plt.subplot(122)
        plt.semilogy(frequenciespat, Sppat)
        plt.xlabel('Частота (Гц)')
        plt.ylabel('Суммарная мощность')
        plt.title('Интегральная мощность по частотам')
        plt.grid(True)
        
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_pat_wavelet_2_{i}.jpeg"), dpi=300)
        plt.close()
        
        try:
           parpat = compute_late_potentials_from_avg(saecgpat, fs1)
            
           # Сохраняем результаты
           with open(os.path.join(save_dir_txt, f"results_2_{i}_pat.txt"), 'w') as f:
                f.write(f"Результаты обработки файла {i}\n")
                f.write("="*50 + "\n")
                f.write(f"Патологический сигнал:\n{parpat}\n")
                f.close()
            
           # Сохраняем параметры в словарь
           if isinstance(parpat, dict):
              file_result['pat_fQRS_ms'] = parpat.get('fQRS_ms', None)
              file_result['pat_RMS40_mkV'] = parpat.get('RMS40_mkV', None)
              file_result['pat_LAS_ms'] = parpat.get('LAS_ms', None)
           elif isinstance(parpat, (list, tuple)) and len(parpat) >= 3:
               file_result['pat_fQRS_ms'] = parpat[0]
               file_result['pat_RMS40_mkV'] = parpat[1]
               file_result['pat_LAS_ms'] = parpat[2]
            
        except:
            pass

        # Добавляем результаты текущего файла в общий список
        results_data.append(file_result)

    # Создаем DataFrame из собранных данных
    df_results = pd.DataFrame(results_data)

    # Переупорядочиваем колонки для удобства чтения
    column_order = [
     'rat_number','norm_fQRS_ms', 'norm_RMS40_mkV', 'norm_LAS_ms', 
     'pat_fQRS_ms', 'pat_RMS40_mkV', 'pat_LAS_ms', 
    
    ]
    # Добавляем только существующие колонки
    existing_columns = [col for col in column_order if col in df_results.columns]
    df_results = df_results[existing_columns]


  
   
    excel_filename = os.path.join(save_dir_xlsx, "ecg_analysisresults2.xlsx")
    df_results.to_excel(excel_filename, index=False)



    file_result = {
    'rat_number': None,  # Явно сохраняем номер крысы
    # Параметры нормального сигнала
    'norm_fQRS_ms': None,
    'norm_RMS40_mkV': None,
    'norm_LAS_ms': None,
    # Параметры патологического сигнала
    'pat_fQRS_ms': None,
    'pat_RMS40_mkV': None,
    'pat_LAS_ms': None
    }
    base_path = "D:/ECG_IAI_RAS/RAT_NEW/"
    file_numbers = range(10, 19)
    results_data = []

    # Создаем прогресс-бар с дополнительной информацией
    pbar = tqdm(
    file_numbers, 
    desc="Обработка файлов", 
    unit="файл",
    ncols=100,  # ширина прогресс-бара
    colour='green',  # цвет (работает в некоторых терминалах)
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
    )
    for i in pbar:
        # Обновляем описание с текущим файлом
        pbar.set_description(f"Обработка файла {i}")
        file_result['rat_number'] =i
        # Здесь ваш код обработки
        print(f"\n{'='*50}")
        print(f"Обработка файла {i}")
        print('='*50)
    
        # Формируем пути к файлам
        file_name = f"{i}_2_Not_filtered.edf"
        file_path = f"{base_path}{i}/2_rat/"
        chanel_d, time, fs1, anot = read_edf(
            file_name, 
            file_path, 
            [0, 1, 2, 3, 4, 5]
        )
        
        pbar.set_postfix(
            file=file_name,
            status="Обработка сигнала..."
        )
        # Дополнительная обработка сигнала
        lf_sig, hf_sig, sig1, fs1, anot = signaladd(
            file_name, 
            file_path
        )
        
        # Применяем винзоризацию
        sig1 = iqr_winsorize(sig1, 5)
        
        
        # Определяем временные метки
        if not anot:
            stabilization_time = 0
            ishemia_time = 1688
            reperfusion_time = 3600
        else:
            stabilization_time = 0
            ishemia_time = next((k for k, v in anot.items() if v == 'Ishemia'), 0)
            reperfusion_time = next((k for k, v in anot.items() if v == 'Reperfusion'), 0)
        
        # Вырезаем участки сигнала
        ynorm = time_slice(
            sig1 - np.mean(sig1),
            stabilization_time + 5,
            stabilization_time + 34,
            fs1
        )
        
        ypat = time_slice(
            sig1 - np.mean(sig1),
            ishemia_time,
            ishemia_time + 34,
            fs1
        )
        
        pbar.set_postfix(
            file=file_name,
            status="Графики сигналов..."
        )
        # Сохраняем графики исходных сигналов
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(ynorm)/fs1, len(ynorm)), ynorm)
        plt.grid(True)
        plt.title(f'Нормальный сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_norm_1_{i}.jpeg"), dpi=300)
        plt.close()
        
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(ypat)/fs1, len(ypat)), ypat)
        plt.grid(True)
        plt.title(f'Патологический сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_pat_1_{i}.jpeg"), dpi=300)
        plt.close()
        
        pbar.set_postfix(
            file=file_name,
            status="Усреднение сигналов..."
        )
        
        # Обработка нормального сигнала
        rpeaks = findpeaks(ynorm, fs1)
        saecgnorm = signalavergedecg(ynorm, fs1, rpeaks)
        
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(saecgnorm)/fs1, len(saecgnorm)), saecgnorm)
        plt.grid(True)
        plt.title(f'Усредненный нормальный сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_norm_1_{i}.jpeg"), dpi=300)
        plt.close()
        
        parnorm = compute_late_potentials_from_avg(saecgnorm, fs1)
        # Сохраняем результаты
        with open(os.path.join(save_dir_txt, f"results_1_{i}_norm.txt"), 'w') as f:
            f.write(f"Результаты обработки файла {i}\n")
            f.write("="*50 + "\n")
          
            f.write(f"Нормальный сигнал:\n{parnorm}\n\n")
            f.close()
        # Сохраняем параметры в словарь
        if isinstance(parnorm, dict):
            file_result['norm_fQRS_ms'] = parnorm.get('fQRS_ms', None)
            file_result['norm_RMS40_mkV'] = parnorm.get('RMS40_mkV', None)
            file_result['norm_LAS_ms'] = parnorm.get('LAS_ms', None)
        elif isinstance(parnorm, (list, tuple)) and len(parnorm) >= 3:
            file_result['norm_fQRS_ms'] = parnorm[0]
            file_result['norm_RMS40_mkV'] = parnorm[1]
            file_result['norm_LAS_ms'] = parnorm[2]
        
        pbar.set_postfix(
            file=file_name,
            status="Вейвлет-анализ (норма)..."
        )
        
        # Вейвлет-анализ нормального сигнала
        Spnorm, frequenciesnorm, powernorm = waveletscaleaogram(saecgnorm, fs1)
        rationorm=calculate_band_power_ratio(powernorm,300,2000)
        with open(os.path.join(save_dir_txt, f"results_ratio_cwt_power_1_{i}_norm.txt"), 'w') as f:
            f.write(f"Результаты обработки файла {i}\n")
            f.write("="*50 + "\n")
          
            f.write(f"Нормальный сигнал:\n{rationorm}\n\n")
            f.close()
        
        plt.figure(figsize=(15,7))
        plt.subplot(121)
        time_axis = np.arange(powernorm.shape[1]) / fs1  
        plt.imshow(powernorm, 
               extent=[time_axis[0], time_axis[-1], frequenciesnorm[-1], frequenciesnorm[0]], 
               cmap='viridis', 
               aspect='auto',
               vmax=abs(powernorm).max(), 
               vmin=-abs(powernorm).max())  
        plt.colorbar(label='Мощность')
        plt.xlabel('Время (с)')
        plt.ylabel('Частота (Гц)')
        plt.title(f'Вейвлет-спектрограмма (норма) - Файл {i}')
        
        plt.subplot(122)
        plt.semilogy(frequenciesnorm, Spnorm)
        plt.xlabel('Частота (Гц)')
        plt.ylabel('Суммарная мощность')
        plt.title('Интегральная мощность по частотам')
        plt.grid(True)
        
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_norm_wavelet_1_{i}.jpeg"), dpi=300)
        plt.close()
        
        pbar.set_postfix(
            file=file_name,
            status="Обработка патологии..."
        )
        
        # Обработка патологического сигнала
        rpeaks = findpeaks(ypat, fs1)
        saecgpat = signalavergedecg(ypat, fs1, rpeaks)
        
        plt.figure(figsize=(15,7))
        plt.plot(np.linspace(0, len(saecgpat)/fs1, len(saecgpat)), saecgpat)
        plt.grid(True)
        plt.title(f'Усредненный патологический сигнал - Файл {i}')
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_pat_1_{i}.jpeg"), dpi=300)
        plt.close()
        
        pbar.set_postfix(
            file=file_name,
            status="Вейвлет-анализ (патология)..."
        )
        
        # Вейвлет-анализ патологического сигнала
        Sppat, frequenciespat, powerpat = waveletscaleaogram(saecgpat, fs1)
        ratiopat=calculate_band_power_ratio(powerpat,300,2000)
        with open(os.path.join(save_dir_txt, f"results_ratio_cwt_power_1_{i}_pat.txt"), 'w') as f:
            f.write(f"Результаты обработки файла {i}\n")
            f.write("="*50 + "\n")
          
            f.write(f"Нормальный сигнал:\n{ratiopat}\n\n")
            f.close()
        
        plt.figure(figsize=(15,7))
        plt.subplot(121)
        time_axis = np.arange(powerpat.shape[1]) / fs1  
        plt.imshow(powerpat, 
               extent=[time_axis[0], time_axis[-1], frequenciespat[-1], frequenciespat[0]], 
               cmap='viridis', 
               aspect='auto',
               vmax=abs(powerpat).max(), 
               vmin=-abs(powerpat).max())  
        plt.colorbar(label='Мощность')
        plt.xlabel('Время (с)')
        plt.ylabel('Частота (Гц)')
        plt.title(f'Вейвлет-спектрограмма (патология) - Файл 1 {i}')
        
        plt.subplot(122)
        plt.semilogy(frequenciespat, Sppat)
        plt.xlabel('Частота (Гц)')
        plt.ylabel('Суммарная мощность')
        plt.title('Интегральная мощность по частотам')
        plt.grid(True)
        
        plt.savefig(os.path.join(save_dir_pic, f"ecg_signal_averged_pat_wavelet_1_{i}.jpeg"), dpi=300)
        plt.close()
        
        try:
           parpat = compute_late_potentials_from_avg(saecgpat, fs1)
            
           # Сохраняем результаты
           with open(os.path.join(save_dir_txt, f"results_1_{i}_pat.txt"), 'w') as f:
                f.write(f"Результаты обработки файла {i}\n")
                f.write("="*50 + "\n")
                f.write(f"Патологический сигнал:\n{parpat}\n")
                f.close()
           # Сохраняем параметры в словарь
           if isinstance(parpat, dict):
              file_result['pat_fQRS_ms'] = parpat.get('fQRS_ms', None)
              file_result['pat_RMS40_mkV'] = parpat.get('RMS40_mkV', None)
              file_result['pat_LAS_ms'] = parpat.get('LAS_ms', None)
           elif isinstance(parpat, (list, tuple)) and len(parpat) >= 3:
               file_result['pat_fQRS_ms'] = parpat[0]
               file_result['pat_RMS40_mkV'] = parpat[1]
               file_result['pat_LAS_ms'] = parpat[2]
            
        except:
            pass

        # Добавляем результаты текущего файла в общий список
        results_data.append(file_result)

    # Создаем DataFrame из собранных данных
    df_results = pd.DataFrame(results_data)

    # Переупорядочиваем колонки для удобства чтения
    column_order = [
     'rat_number','norm_fQRS_ms', 'norm_RMS40_mkV', 'norm_LAS_ms', 
     'pat_fQRS_ms', 'pat_RMS40_mkV', 'pat_LAS_ms', 
    
    ]
    # Добавляем только существующие колонки
    existing_columns = [col for col in column_order if col in df_results.columns]
    df_results = df_results[existing_columns]


  
   
    excel_filename = os.path.join(save_dir_xlsx, "ecg_analysisresults1.xlsx")
    df_results.to_excel(excel_filename, index=False)

  
   
    plt.show()
    
    
    
    
if __name__ == "__main__":
    main()



    
    

